In [1]:
import requests
import os
if not os.path.exists("the-verdict.txt"):
  url = ("https://raw.githubusercontent.com/rasbt/"
 "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
 "the-verdict.txt")
  file_path = "the-verdict.txt"
  response = requests.get(url,timeout=30)
  response.raise_for_status()
  with open(file_path,"wb") as f:
    f.write(response.content)

In [2]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()
print("Total number of characters",len(raw_text))
print(raw_text[:99])

Total number of characters 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re
text = "Hello, world. This -- is? a test."
result = re.split(r'([,.:;?!_()\']|--|\s)',text)
result=[item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', '--', 'is', '?', 'a', 'test', '.']


In [4]:
preprocess=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocess=[item.strip() for item in preprocess if item.strip()]
print(len(preprocess))
print(preprocess[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [5]:
all_words=sorted(set(preprocess))
vocab_size=len(all_words)
print(vocab_size)

1130


In [6]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
class SimpleTokenizerV1:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str={i:s for s,i in vocab.items()}

  def encode(self,text):
    preprocessed=re.split(r'([;:,."?!_()\']|--|\s)',text)
    preprocessed=[
        item.strip() for item in preprocessed if item.strip()
        ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self,ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text

In [10]:
tokenizer=SimpleTokenizerV1(vocab)
text=""""It's the last he painted, you know,"
          Mrs Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 38, 851, 1108, 754, 793, 7]


In [11]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs Gisburn said with pardonable pride.


In [14]:
test_text="Hello, test -- this is."
ids2=tokenizer.encode(test_text)
print(ids2)

KeyError: 'Hello'

In [16]:
all_tokens=sorted(list(set(preprocess)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [17]:
len(vocab.items())

1132

In [24]:
for i,item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [29]:
class SimpleTokenizerV2:
  def __init__(self,vocab):
    self.str_to_int=vocab
    self.int_to_str={i:s for s,i in vocab.items()}
  def encode(self,text):
    preprocessed = re.split(r'[,.:;?_!"()\']|--|\s',text)
    preprocessed=[item.strip() for item in preprocessed if item.strip()]
    preprocessed=[
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self,ids):
    text=" ".join(self.int_to_str[i] for i in ids)
    text=re.sub(r'\s+([.,:;?!"()\'])',r'\1',text)
    return text

In [36]:
tokenizer=SimpleTokenizerV2(vocab)
text1="hello, would you like some tea?"
text2="In the sunlit, terrace of the palace"
text="<|endoftext|> ".join((text1,text2))

In [37]:
tokenizer.encode(text)

[1131, 1120, 1126, 628, 910, 975, 1130, 55, 988, 956, 983, 722, 988, 1131]

In [38]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|> would you like some tea <|endoftext|> In the sunlit terrace of the <|unk|>'

**BytePair encoding**

In [39]:
import importlib
import tiktoken
print("tiktoken version:",importlib.metadata.version("tiktoken"))

tiktoken version: 0.14.0


In [40]:
tokenizer=tiktoken.get_encoding("o200k_base")

In [42]:
text=(
    "Hello do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace <|endoftext|> personally I like coffee and no sugar and matcha coconut boba"
)
integer=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integer)

[13225, 621, 481, 1299, 17966, 30, 220, 199999, 730, 290, 7334, 32758, 173297, 1440, 1236, 33936, 18099, 220, 199999, 21651, 357, 1299, 12525, 326, 860, 15338, 326, 3981, 64, 46672, 287, 14476]


In [43]:
strings=tokenizer.decode(integer)
print(strings)

Hello do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace <|endoftext|> personally I like coffee and no sugar and matcha coconut boba


Tokenizing our book with tiktoken.

In [44]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()
enc_text = tokenizer.encode(raw_text)
print(type(enc_text))
print(len(enc_text))

<class 'list'>
4836


now we will do try to predict the next vector encoding

In [45]:
enc_sample=enc_text[50:]

In [ ]:
context_size=4
x=enc:sample